# Lecture 8: Real-World NLP Pipeline Design
### NLP Course 2027

---

## Learning Outcomes
- Design scalable end-to-end NLP pipelines
- Choose appropriate preprocessing strategies for different tasks
- Handle domain-specific and noisy text
- Apply modular design principles to NLP systems

**Primary Reference:** *Practical NLP* Ch.2 & Ch.3

## 1. The NLP Pipeline

```
Stage 1: Data Collection
    │  Web scraping, APIs, databases, files
    ▼
Stage 2: Data Preprocessing
    │  Cleaning, normalization, tokenization
    ▼
Stage 3: Feature Engineering
    │  Bag-of-words, TF-IDF, embeddings
    ▼
Stage 4: Model Development
    │  Training, validation, hyperparameter tuning
    ▼
Stage 5: Evaluation
    │  Accuracy metrics, error analysis
    ▼
Stage 6: Deployment
    │  API, monitoring, maintenance
```

Each stage feeds into the next. Errors in early stages compound downstream — **garbage in, garbage out**.

*Source: Vajjala et al., Practical NLP, Chapter 2*

## 2. Data Collection Strategies

### Data Sources for NLP
| Source | Method | Considerations |
|--------|--------|----------------|
| Web pages | BeautifulSoup, Scrapy | Rate limiting, robots.txt |
| APIs | requests + JSON | Authentication, quotas |
| Databases | SQL queries | Schema mapping |
| Public datasets | HuggingFace, Kaggle | License, format |
| PDFs/Docs | pdfplumber, python-docx | Layout extraction |
| Social media | Twitter/Reddit APIs | Terms of service |

In [1]:
import re
from urllib.request import urlopen
from html.parser import HTMLParser

class SimpleTextExtractor(HTMLParser):
    """Extract text from HTML, stripping tags."""
    def __init__(self):
        super().__init__()
        self.text_parts = []
        self._skip = False

    def handle_starttag(self, tag, attrs):
        if tag in ('script', 'style', 'head'):
            self._skip = True

    def handle_endtag(self, tag):
        if tag in ('script', 'style', 'head'):
            self._skip = False

    def handle_data(self, data):
        if not self._skip and data.strip():
            self.text_parts.append(data.strip())

    def get_text(self):
        return ' '.join(self.text_parts)

# Simulate HTML content
html_sample = """
<html><head><title>NLP Article</title></head>
<body>
<h1>Introduction to NLP</h1>
<p>Natural Language Processing is a field of AI.</p>
<p>It enables computers to understand human language.</p>
<script>console.log('skip me');</script>
</body></html>
"""

extractor = SimpleTextExtractor()
extractor.feed(html_sample)
print('Extracted text:')
print(extractor.get_text())

Extracted text:
Introduction to NLP Natural Language Processing is a field of AI. It enables computers to understand human language.


## 3. Preprocessing Strategy Decisions

Preprocessing choices **depend on the task**. There's no one-size-fits-all.

| Decision | When to do it | When NOT to |
|----------|--------------|-------------|
| Lowercase | Most classification tasks | Named entity recognition, sentiment with caps |
| Remove stopwords | Topic modeling, IR | Sentiment analysis ("not good"), NER |
| Stemming | Search engines, IR | Tasks needing actual words |
| Lemmatization | Text generation, QA | Speed-critical applications |
| Remove punctuation | BoW classification | Sentence boundary detection |
| Remove numbers | Topic modeling | Financial analysis, date extraction |
| Expand contractions | Sentiment, QA | Speed-critical |

*Source: Practical NLP (Vajjala et al.), Chapter 2*

In [2]:
import re

def expand_contractions(text):
    """Expand common English contractions."""
    contractions = {
        r"can't": 'cannot', r"won't": 'will not', r"don't": 'do not',
        r"doesn't": 'does not', r"didn't": 'did not', r"isn't": 'is not',
        r"aren't": 'are not', r"wasn't": 'was not', r"weren't": 'were not',
        r"haven't": 'have not', r"hasn't": 'has not', r"hadn't": 'had not',
        r"wouldn't": 'would not', r"shouldn't": 'should not',
        r"couldn't": 'could not', r"it's": 'it is', r"i'm": 'i am',
        r"i've": 'i have', r"i'll": 'i will', r"i'd": 'i would',
        r"you're": 'you are', r"they're": 'they are', r"we're": 'we are',
    }
    text = text.lower()
    for contraction, expanded in contractions.items():
        text = re.sub(contraction, expanded, text)
    return text

test = "I can't believe it! They don't understand what's happening. It's not fair."
print('Original: ', test)
print('Expanded: ', expand_contractions(test))

Original:  I can't believe it! They don't understand what's happening. It's not fair.
Expanded:  i cannot believe it! they do not understand what's happening. it is not fair.


In [3]:
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
import nltk

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)

class NLPPipeline:
    """Modular, configurable NLP preprocessing pipeline."""

    def __init__(self, config=None):
        default_config = {
            'expand_contractions': True,
            'lowercase': True,
            'remove_html': True,
            'remove_urls': True,
            'remove_emails': True,
            'remove_numbers': False,
            'remove_punctuation': True,
            'remove_stopwords': True,
            'stem': False,
            'lemmatize': True,
            'min_token_length': 2,
        }
        self.config = {**default_config, **(config or {})}
        self._stop_words = set(stopwords.words('english'))
        self._stemmer = PorterStemmer()
        self._lemmatizer = WordNetLemmatizer()

    def clean(self, text):
        if self.config['remove_html']:
            text = re.sub(r'<[^>]+>', ' ', text)
        if self.config['remove_urls']:
            text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
        if self.config['remove_emails']:
            text = re.sub(r'\S+@\S+', ' ', text)
        if self.config['expand_contractions']:
            text = expand_contractions(text)
        elif self.config['lowercase']:
            text = text.lower()
        if self.config['remove_numbers']:
            text = re.sub(r'\b\d+\b', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    def tokenize(self, text):
        tokens = word_tokenize(text)
        if self.config['remove_punctuation']:
            tokens = [t for t in tokens if t not in string.punctuation]
        if self.config['remove_stopwords']:
            tokens = [t for t in tokens if t not in self._stop_words]
        if self.config['min_token_length']:
            tokens = [t for t in tokens if len(t) >= self.config['min_token_length']]
        return tokens

    def normalize(self, tokens):
        if self.config['stem']:
            return [self._stemmer.stem(t) for t in tokens]
        elif self.config['lemmatize']:
            return [self._lemmatizer.lemmatize(t) for t in tokens]
        return tokens

    def process(self, text):
        cleaned = self.clean(text)
        tokens = self.tokenize(cleaned)
        normalized = self.normalize(tokens)
        return normalized

# Demo
pipeline = NLPPipeline()
noisy = """<p>Check out https://example.com for info! Email: admin@test.org
I can't believe it's working!!! The researchers are studying 3 NLP models."""
result = pipeline.process(noisy)
print(f'Tokens: {result}')

Tokens: ['check', 'info', 'email', 'believe', 'working', 'researcher', 'studying', 'nlp', 'model']


In [4]:
# Comparing preprocessing strategies for different tasks
sample = "She wasn't happy about the delayed flight. The service was NOT good!"

strategies = {
    'Sentiment (preserve negation)': NLPPipeline({'remove_stopwords': False, 'expand_contractions': True}),
    'Topic Modeling (aggressive)': NLPPipeline({'remove_stopwords': True, 'remove_numbers': True}),
    'Search Index (stem)': NLPPipeline({'stem': True, 'lemmatize': False}),
}

print(f'Input: {sample}\n')
for strategy_name, pipe in strategies.items():
    result = pipe.process(sample)
    print(f'{strategy_name}:')
    print(f'  {result}\n')

Input: She wasn't happy about the delayed flight. The service was NOT good!

Sentiment (preserve negation):
  ['she', 'wa', 'not', 'happy', 'about', 'the', 'delayed', 'flight', 'the', 'service', 'wa', 'not', 'good']

Topic Modeling (aggressive):
  ['happy', 'delayed', 'flight', 'service', 'good']

Search Index (stem):
  ['happi', 'delay', 'flight', 'servic', 'good']



## 4. Handling Special Text Types

### Social Media Text
- Abbreviations: `lol`, `omg`, `imo`, `imo`, `tbh`
- Hashtags: `#NLP` — might want to keep or split: `NLP`
- Mentions: `@user` — strip or keep as feature
- Emojis: 😊 → `happy_face` or strip

### Domain-Specific Text
- **Medical**: abbreviations (`Dx=diagnosis`), acronyms (`COPD`, `MI`)
- **Legal**: complex sentences, Latin terms
- **Scientific**: citations, chemical formulas

### Multi-language Text
- Language detection first: `langdetect` or `fastText`
- Language-specific preprocessing

In [5]:
import re

def preprocess_tweet(tweet, keep_hashtags=True, keep_mentions=False):
    """Specialized preprocessing for Twitter/social media text."""
    # Expand common abbreviations
    abbrevs = {'u': 'you', 'r': 'are', 'ur': 'your', 'gr8': 'great',
               'thx': 'thanks', 'btw': 'by the way', 'imo': 'in my opinion'}
    # Remove URLs
    tweet = re.sub(r'https?://\S+', '', tweet)
    # Handle hashtags
    if keep_hashtags:
        tweet = re.sub(r'#(\w+)', r'\1', tweet)  # remove # but keep word
    else:
        tweet = re.sub(r'#\w+', '', tweet)
    # Handle mentions
    if not keep_mentions:
        tweet = re.sub(r'@\w+', '', tweet)
    # Remove emojis (basic approach)
    tweet = tweet.encode('ascii', 'ignore').decode('ascii')
    # Expand abbreviations
    words = tweet.lower().split()
    words = [abbrevs.get(w, w) for w in words]
    tweet = ' '.join(words)
    return re.sub(r'\s+', ' ', tweet).strip()

tweets = [
    "@JohnDoe great job on #NLP! Can't believe u did it lol https://example.com",
    "#AI is taking over! imo we r all doomed 😱 btw thx for the update!"
]
for tweet in tweets:
    print(f'Original:  {tweet}')
    print(f'Processed: {preprocess_tweet(tweet)}')
    print()

Original:  @JohnDoe great job on #NLP! Can't believe u did it lol https://example.com
Processed: great job on nlp! can't believe you did it lol

Original:  #AI is taking over! imo we r all doomed 😱 btw thx for the update!
Processed: ai is taking over! in my opinion we are all doomed by the way thanks for the update!



In [6]:
# Data augmentation for NLP
def synonym_augmentation(text, replacement_rate=0.2):
    """
    Simple data augmentation: replace random words with synonyms.
    Uses WordNet for synonyms.
    """
    from nltk.corpus import wordnet as wn
    import random
    nltk.download('wordnet', quiet=True)

    words = word_tokenize(text)
    augmented = []
    for word in words:
        if random.random() < replacement_rate and word.isalpha():
            synsets = wn.synsets(word.lower())
            if synsets:
                synonyms = [
                    lemma.name() for syn in synsets[:3]
                    for lemma in syn.lemmas()
                    if lemma.name().lower() != word.lower() and '_' not in lemma.name()
                ]
                if synonyms:
                    replacement = random.choice(synonyms)
                    augmented.append(replacement)
                    continue
        augmented.append(word)
    return ' '.join(augmented)

original = 'The researchers published an interesting paper about machine learning.'
print(f'Original:  {original}')
for i in range(3):
    augmented = synonym_augmentation(original)
    print(f'Augmented: {augmented}')

Original:  The researchers published an interesting paper about machine learning.
Augmented: The investigator published an interesting paper about machine learning .
Augmented: The researchers published an interesting paper around machine eruditeness .
Augmented: The researchers published an interesting paper about machine learning .


## 5. Pipeline Evaluation

Before deploying, evaluate your pipeline:

```python
# What to measure at each stage:
Stage 2 (Preprocessing): sample + human review
    → Are important words preserved?
    → Is noise removed correctly?

Stage 4 (Model): hold-out test set
    → Accuracy, F1, BLEU, etc.

Stage 6 (Production): A/B testing
    → Latency, throughput, user satisfaction
```

## Practice Exercises

See **`Lecture-08-Homework.ipynb`** for the practice exercises accompanying this lecture.

In [7]:
# Pipeline quality check
def quality_check(original_texts, processed_tokens_list):
    """Run basic quality checks on preprocessing output."""
    total = len(original_texts)
    too_short = sum(1 for t in processed_tokens_list if len(t) < 3)
    empty = sum(1 for t in processed_tokens_list if len(t) == 0)
    avg_tokens = sum(len(t) for t in processed_tokens_list) / total
    orig_avg_words = sum(len(t.split()) for t in original_texts) / total
    retention = avg_tokens / orig_avg_words if orig_avg_words else 0

    print('Pipeline Quality Report')
    print('-' * 40)
    print(f'Total documents: {total}')
    print(f'Empty after processing: {empty}')
    print(f'Too short (<3 tokens): {too_short}')
    print(f'Avg original words: {orig_avg_words:.1f}')
    print(f'Avg processed tokens: {avg_tokens:.1f}')
    print(f'Token retention rate: {retention:.2%}')

texts = [
    'The quick brown fox jumps over the lazy dog.',
    'NLP with Python is an excellent resource for learning.',
    'I love natural language processing and machine learning!',
    'Data science applications span many industries today.',
]
pipe = NLPPipeline()
processed = [pipe.process(t) for t in texts]
quality_check(texts, processed)

Pipeline Quality Report
----------------------------------------
Total documents: 4
Empty after processing: 0
Too short (<3 tokens): 0
Avg original words: 8.2
Avg processed tokens: 6.0
Token retention rate: 72.73%


## Summary

| Stage | Key Decision | Tool |
|-------|-------------|------|
| Collection | Scraping vs API vs dataset | BeautifulSoup, requests |
| Cleaning | What noise to remove | regex |
| Tokenization | Which tokenizer fits domain | NLTK, spaCy |
| Normalization | Stem vs lemmatize | NLTK stemmers |
| Feature Eng | BoW vs TF-IDF vs embeddings | scikit-learn |

Pipeline design is **iterative**: start simple, measure, improve.

**Next Lecture**: Text Representation & Embeddings — vectors, Word2Vec, distributed representations.

---
*Book references: Practical NLP Ch.2, 3 | NLP with Python Ch.3*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**